In [ ]:
from torch.nn import *
import torch
import numpy as np

torch.set_default_tensor_type(torch.cuda.FloatTensor)
M = RNN(5, 16)
m = Sequential(Linear(20, 32), Hardswish(1), Linear(32, 2))
o = torch.optim.Adam((*M.parameters(), *m.parameters()))
# M, m, o = torch.load("nnTrainerEvaler-Copy1.model.pk")

In [ ]:
from sklearn.datasets import make_classification
import matplotlib.pyplot as plt


def axy():
    x = np.random.randint(1, 5)
    a = [
        make_classification(
            1024,
            4,
            n_informative=x,
            n_redundant=np.random.randint(0, 5 - x),
            n_clusters_per_class=np.random.randint(1, 2 ** (x - 1) + 1),
        )
        for a in range(4)
    ]
    x = torch.tensor(np.stack([a[0] for a in a], 1), dtype=torch.float)
    y = torch.tensor(np.stack([a[1] for a in a], 1), dtype=torch.long)
    return torch.cat((x, y[..., None].float()), 2), x, y


def train(M, a):
    return M(a)[0]


def fast_los(m, d, x, y):
    k = torch.randint(len(d), d.shape[:1]) % torch.arange(1, len(d) + 1)
    return functional.cross_entropy(
        m(torch.cat((d, x[k]), 2)).view(-1, 2), y[k].ravel()
    )


def slow_los(m, d, x, y):
    return sum(
        functional.cross_entropy(
            m(
                torch.cat((d[i].broadcast_to((i + 1,) + d.shape[1:]), x[: i + 1]), 2)
            ).view(-1, 2),
            y[: i + 1].ravel(),
        )
        for i in range(len(d))
    )


def full_los(m, d, x, y):
    return functional.cross_entropy(
        m(
            torch.cat(
                (
                    d[:, None].broadcast_to((len(d),) + d.shape),
                    x[None].broadcast_to((len(x),) + x.shape),
                ),
                3,
            )
        ).view(-1, 2),
        y.ravel().repeat(len(y)),
    )


def slow_acc(m, d, x, y):
    return [
        (
            m(
                torch.cat((d[i].broadcast_to((i + 1,) + d.shape[1:]), x[: i + 1]), 2)
            ).argmax(2)
            == y[: i + 1]
        )
        .half()
        .mean()
        .item()
        for i in range(len(d))
    ]


def full_acc(m, d, x, y):
    return (
        (
            m(
                torch.cat(
                    (
                        d[:, None].broadcast_to((len(d),) + d.shape),
                        x[None].broadcast_to((len(x),) + x.shape),
                    ),
                    3,
                )
            ).argmax(3)
            == y
        )
        .half()
        .mean((1, 2))
        .to("cpu", torch.float)
    )


while 1:
    for i in range(20):
        for i in range(200):
            a, x, y = axy()
            d = train(M, a)
            i = full_los(m, d, x, y)
            i.backward()
            o.step()
            o.zero_grad()
        print(i)
    plt.figure()
    plt.plot(full_acc(m, d, x, y))
    plt.show()